In this document I aggregate scripts used in data spliting and training already existing models for the project 

# .arrow file converter

In [ ]:
from datasets import Dataset
from PIL import Image
import os
import glob

arrow_folder = "C:\\Users\\Komputer\\Desktop\\studia-Informatyka\\Projekt grupowy\\Pobranie danych i wstępny model\\prithivMLmods___ai-vs-deepfake-vs-real\\default\\0.0.0\\c3f02b29cf666976b056fd04a4332229ded0477a"
output_root = "images"

global_index = 0

arrow_files = glob.glob(os.path.join(arrow_folder, "*.arrow"))

for arrow_path in arrow_files:
    print(f"Przetwarzam: {arrow_path}")
    dataset = Dataset.from_file(arrow_path)

    for item in dataset:
        img = item["image"]
        label = str(item["label"]) 

        label_folder = os.path.join(output_root, label)
        os.makedirs(label_folder, exist_ok=True)

        save_path = os.path.join(label_folder, f"img_{global_index}.jpg")

        if isinstance(img, dict):
            img = Image.open(img["path"])

        img.save(save_path)
        global_index += 1



# Data split

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

source_dir = "dataset"
output_dir = "split_data"
test_ratio = 0.2  

classes = ["AI", "Real"]

for cls in classes:
    img_dir = os.path.join(source_dir, cls)
    images = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    train_imgs, test_imgs = train_test_split(images, test_size=test_ratio, random_state=42)

    for split, split_imgs in zip(["train", "test"], [train_imgs, test_imgs]):
        split_dir = os.path.join(output_dir, split, cls)
        os.makedirs(split_dir, exist_ok=True)

        for img in split_imgs:
            src = os.path.join(img_dir, img)
            dst = os.path.join(split_dir, img)
            shutil.copy2(src, dst)

# EfficientNetB3 training

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import timm


data_dir = "split_data"
batch_size = 30
epochs = 2
num_classes = 2
model_path = "efficientnetb3_ai_classifier.pth"

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(300),                  
    transforms.RandomHorizontalFlip(),                 
    transforms.RandomRotation(10),                     
    transforms.ColorJitter(brightness=0.1, 
                           contrast=0.1, 
                           saturation=0.1, 
                           hue=0.05),                
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(300),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model("efficientnet_b3", pretrained=True, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


for epoch in range(epochs):
    print("Aktualna epoka:", epoch )
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

model.eval()
y_true = []
y_pred = []
y_prob = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        _, preds = torch.max(outputs, 1)

        probs = torch.softmax(outputs, dim=1)[:, 1]

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

accuracy = 100 * (sum([p == t for p, t in zip(y_pred, y_true)]) / len(y_true))
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"AUC: {auc:.2f}")

dummy_input = torch.randn(1, 3, 300, 300).to(device) 
torch.save(model.state_dict(), model_path)
print(f"Model zapisany jako {model_path}")

torch.onnx.export(
    model,                       
    dummy_input,                 
    "model_efficientnet_b3.onnx",               
    export_params=True,        
    opset_version=11,            
    do_constant_folding=True,    
    input_names=["input"],     
    output_names=["output"],     
    dynamic_axes={               
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("Model zapisany jako model.onnx")

# Resnet50 training 

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
from sklearn.metrics import roc_auc_score, precision_score, recall_score

data_dir = "split_data"
batch_size = 40
epochs = 2
num_classes = 2
model_path = "resnet50_ai_classifier.pth"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(epochs):
    print("Aktualna epoka:", epoch )
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

model.eval()
y_true = []
y_pred = []
y_prob = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        _, preds = torch.max(outputs, 1)

        probs = torch.softmax(outputs, dim=1)[:, 1]

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

accuracy = 100 * (sum([p == t for p, t in zip(y_pred, y_true)]) / len(y_true))
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"AUC: {auc:.2f}")

dummy_input = torch.randn(1, 3, 224, 224).to(device)  

torch.save(model.state_dict(), model_path)
print(f"Model zapisany jako {model_path}")

torch.onnx.export(
    model,                       
    dummy_input,                 
    "model_resnet_50.onnx",               
    export_params=True,         
    opset_version=11,            
    do_constant_folding=True,    
    input_names=["input"],       
    output_names=["output"],     
    dynamic_axes={               
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("Model zapisany jako model.onnx")

# Vision_TransformerB16 training

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import timm


data_dir = "split_data"
batch_size = 30
epochs = 2
num_classes = 2
model_path = "vit_b16_ai_classifier.pth"

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),              
    transforms.RandomHorizontalFlip(),              
    transforms.RandomRotation(10),                     
    transforms.ColorJitter(brightness=0.1, 
                           contrast=0.1, 
                           saturation=0.1, 
                           hue=0.05),                  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model("efficientnet_b3", pretrained=True, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(epochs):
    print("Aktualna epoka:", epoch )
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

model.eval()
y_true = []
y_pred = []
y_prob = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        _, preds = torch.max(outputs, 1)

        probs = torch.softmax(outputs, dim=1)[:, 1]

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

accuracy = 100 * (sum([p == t for p, t in zip(y_pred, y_true)]) / len(y_true))
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"AUC: {auc:.2f}")

dummy_input = torch.randn(1, 3, 224, 224).to(device) 

torch.save(model.state_dict(), model_path)
print(f"Model zapisany jako {model_path}")
torch.onnx.export(
    model,                       
    dummy_input,                 
    "Vision_Transformer_B16.onnx",               
    export_params=True,        
    opset_version=11,           
    do_constant_folding=True,   
    input_names=["input"],     
    output_names=["output"],     
    dynamic_axes={              
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("Model zapisany jako model.onnx")